# Topic: Multi-Echelon Automotive Supply Chain Knowledge Graph

**Students:**  
- Bekithemba Nkomo  
- Masheia Dzimba
- Peter Mangoro


This project will use Neo4j to model a multi-echelon automotive production network as a knowledge graph and then apply Cypher and Neo4j Graph Data Science (GDS) to analyze dependency structure, bottlenecks, demand propagation, and operational risk.

The primary dataset is:

- Moetz, A., Quetschlich, M., & Otto, B. (2020). *Data for: Optimisation model for multi-item multi-echelon supply chains with nested multi-level products* (Version 1) [Data set]. Mendeley Data. [https://doi.org/10.17632/pr3sdy5vp3.1](https://doi.org/10.17632/pr3sdy5vp3.1)

The dataset describes an automotive production network with one OEM, four first-tier suppliers, and two second-tier suppliers, together with their arcs, production and logistics processes, product structures, inventories, and customer demand over time.

The desired outcome is an interpretable, network-based view of how product dependencies and node-to-node flows shape supply-chain performance, with questions such as:

- Which products have the most complex upstream dependency chains?
- Which nodes and arcs are the most operationally critical?
- Where are bottlenecks and single points of failure concentrated?
- How does downstream customer demand propagate upstream through the network?

The solution will be implemented in Neo4j (Desktop), queried in Cypher, and analyzed with at least one GDS workflow in a Jupyter Notebook deliverable.

---
## Impact

- **Operational impact:** Supports better visibility into product dependencies, constrained arcs, and critical nodes that affect production continuity and demand fulfillment.
- **Business impact:** Helps identify where disruption risk is concentrated, where inventory and capacity pressure may emerge, and which parts of the network should be prioritized for mitigation.
- **Analytical impact:** Demonstrates how graph modeling captures multi-hop supply-chain structure more naturally than flat tables, especially for bill-of-material relationships, demand back-propagation, and bottleneck analysis.
- **Academic impact:** Provides a strong example of how knowledge graphs, Cypher, and GDS can be used to analyze a true multi-echelon supply chain rather than a simple order-fulfillment dataset.

---
## Project Goals

- **Project goal:** Create a Neo4j knowledge graph for a multi-echelon automotive supply chain and produce an analytic notebook with interpretable insights about dependency structure, bottlenecks, inventory position, and demand propagation.
- **Learning goal:** Demonstrate competence in graph modeling, Cypher-based graph EDA, and at least one GDS algorithm applied to a supply-chain network.
- **Capstone goal:** Satisfy the self-defined project requirements by using a self-selected dataset, answering at least 8 graph EDA questions, and completing the Option 2 analytical path with 2 deeper Cypher questions plus 1 GDS-based question.

---
## Literature Review / Environmental Scan / Market Research

- [Neo4j Supply Chain (Pharma) Demo](https://neo4j.com/developer/demos/supply_chain-demo/) for reference graph concepts such as supplier relationships, product flow, traceability, demand back-propagation, and bottleneck identification.
- [Mendeley Data – Automotive production network dataset](https://data.mendeley.com/datasets/pr3sdy5vp3/1) as the primary data source for this project.
- Neo4j Graph Data Science documentation for graph projection and centrality analysis.
- Neo4j Cypher documentation for graph traversal, path analysis, and relationship-oriented querying.
- General supply-chain analytics literature on multi-echelon systems, bill of materials, capacity constraints, and demand propagation.

The environmental scan suggests that graph databases are especially useful for supply-chain problems when the analysis requires tracing multi-hop dependencies, identifying bottlenecks, and reasoning across interconnected production and logistics stages.

---
## Datasets

- **Primary dataset (used in this project):** Moetz, A., Quetschlich, M., & Otto, B. (2020). *Data for: Optimisation model for multi-item multi-echelon supply chains with nested multi-level products* (Version 1) [Data set]. Mendeley Data. [https://doi.org/10.17632/pr3sdy5vp3.1](https://doi.org/10.17632/pr3sdy5vp3.1)

  **Description:** The dataset describes an automotive production network composed of one OEM, four first-tier suppliers, and two second-tier suppliers, together with the nodes, arcs, production and logistics processes, bill-of-material structures, inventories, and customer demand needed to study a multi-echelon supply chain.

  **Included data elements:**
  - product structure of 28,049 products
  - 12 supply-chain nodes
  - 11 arcs with lead-time and capacity information
  - bill of materials
  - initial inventories and flows
  - customer demand over a 14-day horizon

  **Workbook tabs used for graph design:** `products`, `nodes`, `nodes_inflow`, `arcs`, `capacity_at_arc`, `max_flow_product_per_arc`, `max_flow_group_per_arc`, `operations`, `BOM`, `demands`, `initial_inventories`, and `initial_flows`


## Domain and Terminology

This project uses several supply-chain terms that are important to define clearly.

- **OEM (Original Equipment Manufacturer):** The primary manufacturer responsible for assembling and delivering the final product. The published dataset description states that the network includes one OEM, but the workbook tabs themselves appear to encode network entities mainly as generic node identifiers. For that reason, OEM is treated in this project as a possible derived role on `Node`, not as a guaranteed raw node type from the workbook.
- **First-tier supplier:** A supplier that provides parts, assemblies, or materials directly to the OEM. In this project, supplier tier is treated as an analytical classification that may later be inferred or assigned as a node property if defensible.
- **Second-tier supplier:** A supplier that provides parts or materials to first-tier suppliers rather than directly to the OEM. As with first-tier suppliers, this is treated as a role or tier classification rather than an explicitly observed workbook entity.
- **Multi-echelon supply chain:** A supply chain with multiple levels of suppliers, producers, and downstream demand nodes rather than a single supplier-to-customer link.
- **Bill of Materials (BOM):** A structured description of which component products are required to build a higher-level product.
- **Arc:** A directed connection between two supply-chain nodes representing logistics or production flow, often with lead time and capacity.
- **Demand back-propagation:** Reasoning backward from downstream customer demand to determine which upstream nodes, products, and arcs are required to fulfill it.

These terms help position the project as a graph analysis of a true multi-echelon supply chain rather than a sales-only or shipping-only dataset while keeping the graph model grounded in what is directly supported by the workbook.

## Proposed Neo4j Graph Model

### Node Types

| Node Type | Meaning in the Supply Chain | Key Properties |
|---|---|---|
| `Node` | A supply-chain location or operational entity in the network, such as a supplier inventory point, production point, or downstream node. | `nodeId`,`role`, `tier` |
| `Product` | A concrete product or component identifier, including finished vehicles and intermediate parts such as engines or gears. | `productId`, `transportationSize`, `isFinishedGood`, `isComponent`, `bomDepth` |
| `ProductGroup` | A broader grouping of products, such as `car`, `engine`, `gear`, or another class used in the workbook. | `groupName` |
| `Period` | A planning time step in the demand and flow horizon. | `periodId` |
| `DemandFact` | One demand line: a `Node` requests a `Product` quantity in a `Period` (via `HAS_DEMAND`, `FOR_PRODUCT`, `IN_PERIOD`). | `demandKey` (unique), `quantity` |

### Relationship Types

| Relationship Type | Meaning in the Supply Chain | Key Properties |
|---|---|---|
| `BELONGS_TO` | Connects a `Product` to its `ProductGroup`. | no required properties |
| `REQUIRES` | Represents bill-of-material dependency, where a higher-level product requires a lower-level component product. | `quantity` |
| `SHIPS_TO` | Represents a directed arc between supply-chain nodes. | `leadTime`, `productGroup` |
| `HANDLES` | Indicates that a node can receive, store, or process a product. | no required properties |
| `HAS_DEMAND` | Links a supply-chain `Node` to a `DemandFact` it originates. | no required properties |
| `FOR_PRODUCT` | Links a `DemandFact` to the demanded `Product`. | no required properties |
| `IN_PERIOD` | Links a `DemandFact` to its planning `Period`. | no required properties |
| `HOLDS` | Represents inventory of a product at a node. | `initialInventory`, `safetyStock`, `maxInventory`, `period` |
| `INITIAL_FLOW_TO` | Represents product-specific flow already in motion at the start of the planning horizon. | `product`, `period`, `initialFlow` |
| `TRANSFORMS` | Represents production or transformation logic at a node, mapping input groups to output groups. | `inputGroup`, `outputGroup`, `inputQty`, `outputQty`, `alpha`, `beta` |
| `PLANNED_FLOW_TO` | Represents product-specific planned or maximum flow along an arc in a given period. | `product`, `period`, `plannedFlow` |
| `GROUP_FLOW_TO` | Represents product-group-specific planned or maximum flow along an arc in a given period. | `productGroup`, `period`, `plannedFlow` |

### Derived Properties and Analytical Labels

| Property / Label | Attached To | Meaning |
|---|---|---|
| `role` | `Node` | A derived interpretation such as OEM, supplier, plant, inventory node, or demand node. |
| `tier` | `Node` | A derived supply-tier classification, such as OEM, first-tier supplier, or second-tier supplier, only if a defensible mapping is identified. |
| `leadTime` | `SHIPS_TO` | Time needed for movement or process completion between nodes. |
| `capacity` | Arc-related relationships | Time-dependent transport or production capacity. |
| `quantity` | `REQUIRES`, `DemandFact`, `INITIAL_FLOW_TO` | Amount of component requirement, demand (on `DemandFact`), or flow. |
| `inputGroup` / `outputGroup` | `TRANSFORMS` | Product-group structure of node-level production or transformation activity. |
| `bomDepth` | `Product` | A derived analytical property representing how deep a product is in the dependency chain. |
| `isFinishedGood` | `Product` | Indicates whether a product behaves like a finished output in the supply-chain graph. |
| `isComponent` | `Product` | Indicates whether a product behaves like an intermediate or lower-level component. |

---
## Design Concepts

The design is a knowledge graph that models a multi-echelon automotive supply chain as a connected network of products, supply-chain nodes, transformation processes, inventory positions, and downstream demand.

Rather than treating each spreadsheet row as an isolated record, the graph emphasizes:

- upstream and downstream dependency chains
- product-component structure through BOM relationships
- movement across node-to-node arcs
- time-aware demand, inventory, and flow
- structurally important nodes and routes in the network

This design supports network-style insights such as:

- dependency concentration in particular products or nodes
- bottleneck concentration in specific arcs or intermediate nodes
- propagation of downstream demand into upstream product requirements
- identification of structurally central nodes using GDS

The output will include Cypher-based EDA, two deeper analytical Cypher questions, and one GDS-based analysis with narrative interpretation.

---
## Conceptual Architecture

### Services/tools

- Neo4j Desktop or Neo4j AuraDB
- Neo4j Browser and Cypher
- APOC for schema exploration and utility procedures
- Neo4j Graph Data Science (GDS)
- Jupyter Notebook for reproducible documentation and output capture
- Python for preprocessing the `.xlsb` workbook into importable tables if needed

### Data + database considerations

The dataset is distributed as an Excel binary workbook, so preprocessing will likely be required before import. The graph implementation should preserve the distinction between:

- structural entities such as nodes and products
- dependency edges such as BOM relationships
- logistics/process edges such as arcs and initial flows
- time-varying facts such as capacity and demand by period

The architecture should keep the graph simple enough for Cypher and GDS while still preserving the multi-echelon structure that makes the dataset academically valuable.

### Processing/server architecture

- Local single-node Neo4j instance is sufficient because the node and arc counts are moderate even though the BOM and demand tables are large.
- Use a staged import pipeline and batching to avoid memory spikes.
- Preserve workbook-sheet provenance so relationships and properties can be traced back to their source tables during analysis.

---
## Analytical Design

### Metrics and signals

The planned analysis will focus on signals that are meaningful in a multi-echelon supply-chain graph:

- **Dependency complexity:** number and depth of upstream `REQUIRES` paths for each finished product
- **Node criticality:** number of inbound and outbound arcs per node and node centrality in the logistics network
- **Arc pressure:** relationship between arc lead time, capacity, and downstream demand exposure
- **Inventory position:** initial inventory and maximum inventory relative to node-product demand requirements
- **Demand propagation:** how downstream demand can be traced back to upstream products and nodes
- **Flow dependency:** which products and groups dominate initial flows and constrained movement paths

### Audience

- a business-style audience interested in bottlenecks, operational concentration, and resilience
- supply chain analysts or operations leadership interested in mitigation planning

### Outputs and display

- ranked lists of critical nodes, arcs, and products
- path-style interpretations of upstream dependency chains
- tables or plots summarizing demand, inventory, and flow concentration
- GDS outputs showing central nodes in the supply network

---
## Sample Question Set

### EDA

The proposal will define and answer at least 8 graph EDA questions, such as:

1. How many nodes, products, product groups, and periods are represented in the graph?
2. Which product groups contain the most products?
3. Which products have the largest number of direct BOM dependencies?
4. Which products have the deepest upstream component chains?
5. Which nodes have the highest number of inbound and outbound `SHIPS_TO` relationships?
6. Which nodes are associated with the greatest number of demand records?
7. Which products appear in the largest number of demand periods?
8. Which nodes begin with the largest inventory or initial-flow positions?

### Deeper analytical questions

Under Option 2 of the project guidelines, the project will also define and answer:

- **Deeper Cypher Question 1:** Which finished products depend on the deepest or broadest upstream component chains, and what does that imply about supply complexity and disruption exposure?
- **Deeper Cypher Question 2:** Which arcs are the most operationally critical when demand, lead times, and capacity limits are considered together, and where do likely bottlenecks emerge?
- **GDS Question:** Which nodes are most structurally central to the automotive supply chain network?

### Planned GDS algorithm

The primary GDS algorithm proposed is **Betweenness Centrality** on a projected `Node` graph using `SHIPS_TO` relationships, because it is well suited for identifying intermediary bottlenecks in a supply network.

---
## Data Exchange/Processing Framework

### Processing steps

1. Extract the relevant workbook tabs from the `.xlsb` file into importable tabular files.
2. Validate schema and data types for products, nodes, arcs, BOM records, operations, demand, inventory, and flow.
3. Ingest into Neo4j using a staged import process:
   - create `Node`, `Product`, `ProductGroup`, and `Period` entities
   - create BOM, arc, inventory, demand, and transformation relationships
   - assign derived properties such as `role`, `tier`, `leadTime`, and `quantity` where appropriate
4. Run graph EDA queries to verify graph structure, node counts, and relationship distributions.
5. Run the two deeper Cypher analyses and capture outputs with explanatory markdown.
6. Create a GDS in-memory graph projection for the `Node` network and run Betweenness Centrality.
7. Interpret the results in the context of supply-chain bottlenecks, dependency concentration, and demand fulfillment.

### Data flow

`Workbook tabs → extracted CSVs → Neo4j (constraints + ingestion) → Cypher EDA/analytics → GDS projection/algorithm → Notebook interpretation`

---
## References and Hyperlinks

- Moetz, A., Quetschlich, M., & Otto, B. (2020). *Data for: Optimisation model for multi-item multi-echelon supply chains with nested multi-level products* (Version 1) [Data set]. Mendeley Data. [https://doi.org/10.17632/pr3sdy5vp3.1](https://doi.org/10.17632/pr3sdy5vp3.1)
- [Mendeley Data landing page for the automotive production network dataset](https://data.mendeley.com/datasets/pr3sdy5vp3/1)
- [Neo4j Supply Chain (Pharma) Demo](https://neo4j.com/developer/demos/supply_chain-demo/)
- [Neo4j Graph Data Science docs](https://neo4j.com/docs/graph-data-science/current/)
- [Neo4j Cypher manual](https://neo4j.com/docs/cypher-manual/current/)
- [Barabási, A.-L. – Network Science](http://networksciencebook.com/)

